In [3]:
!pip install -q openai pandas openpyxl

In [4]:
import pandas as pd
import time
import re
from openai import OpenAI
from google.colab import files

In [6]:
uploaded = files.upload()

input_file = list(uploaded.keys())[0]

df = pd.read_csv(input_file)

print("File loaded:", input_file)
print("Columns:", df.columns.tolist())
print("Number of questions:", len(df))


if "question" not in df.columns:
    raise ValueError(
        "Column 'question' not found. Available columns: "
        + str(df.columns.tolist())
    )



if "answer" not in df.columns:
    raise ValueError(
        "Accuracy cannot be calculated because an 'answer' column "
        "was not found.\nAvailable columns: "
        + str(df.columns.tolist())
    )



Saving gsm8k_style_50_questions_answers.csv to gsm8k_style_50_questions_answers (1).csv
File loaded: gsm8k_style_50_questions_answers (1).csv
Columns: ['question_id', 'question', 'answer']
Number of questions: 50


In [7]:
API_KEY = "key"

client = OpenAI(api_key=API_KEY)

In [8]:
def extract_number(text):

    # Remove commas from numbers such as 1,000
    text = text.replace(",", "")

    # Look for common answer patterns first
    patterns = [
        r"(?:final answer|answer)\s*(?:is|:)?\s*\$?(-?\d+(?:\.\d+)?)",
        r"####\s*\$?(-?\d+(?:\.\d+)?)"
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)

        if matches:
            return matches[-1]

    # If no explicit answer is found,
    # take the last number in the response
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text)

    if numbers:
        return numbers[-1]

    return None

In [9]:
def ask_o3_mini(question):

    prompt = f"""
Solve the following GSM8K math problem.

Show your reasoning briefly and then provide the final answer.

At the very end, write the answer in this exact format:

FINAL ANSWER: <number>

Question:
{question}
"""

    start_time = time.perf_counter()

    response = client.responses.create(
        model="o3-mini",
        input=prompt
    )

    end_time = time.perf_counter()

    response_time = end_time - start_time

    full_response = response.output_text

    model_answer = extract_number(full_response)

    return full_response, model_answer, response_time

##

results = []

for i, row in df.iterrows():

    question_no = i + 1
    question = str(row["question"])
    correct_answer = str(row["answer"]).strip()

    print(f"\nProcessing {question_no}/{len(df)}")

    try:

        full_response, model_answer, response_time = ask_o3_mini(question)

        # Normalize answers before comparison
        correct_normalized = correct_answer.replace(",", "").strip()
        model_normalized = str(model_answer).replace(",", "").strip()

        # Compare numerically where possible
        try:
            is_correct = float(model_normalized) == float(correct_normalized)
        except:
            is_correct = model_normalized == correct_normalized

        results.append({
            "Question_No": question_no,
            "Question": question,
            "Correct_Answer": correct_answer,
            "Model": "o3-mini",
            "Model_Answer": model_answer,
            "Is_Correct": is_correct,
            "Response_Time_Seconds": round(response_time, 3),
            "Full_Response": full_response
        })

        print("Correct Answer :", correct_answer)
        print("Model Answer   :", model_answer)
        print("Correct?       :", is_correct)
        print("Time           :", round(response_time, 3), "seconds")

    except Exception as e:

        results.append({
            "Question_No": question_no,
            "Question": question,
            "Correct_Answer": correct_answer,
            "Model": "o3-mini",
            "Model_Answer": None,
            "Is_Correct": False,
            "Response_Time_Seconds": None,
            "Full_Response": f"ERROR: {str(e)}"
        })

        print("ERROR:", e)


# ==

results_df = pd.DataFrame(results)


# ===

total_questions = len(results_df)

correct_count = results_df["Is_Correct"].sum()

incorrect_count = total_questions - correct_count

accuracy = (correct_count / total_questions) * 100

average_time = results_df["Response_Time_Seconds"].mean()


# ====

print("\n")
print("=" * 60)
print("             GSM8K MODEL SCORECARD")
print("=" * 60)

print(f"Model              : o3-mini")
print(f"Total Questions    : {total_questions}")
print(f"Correct Answers    : {correct_count}")
print(f"Incorrect Answers  : {incorrect_count}")
print(f"Accuracy           : {accuracy:.2f}%")
print(f"Average Response   : {average_time:.3f} seconds")

print("=" * 60)


# ====

output_file = "gsm8k_o3_mini_results.csv"

results_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


# ====

display(results_df)

files.download(output_file)



Processing 1/50
Correct Answer : 32
Model Answer   : 32
Correct?       : True
Time           : 8.032 seconds

Processing 2/50
Correct Answer : 41
Model Answer   : 41
Correct?       : True
Time           : 1.464 seconds

Processing 3/50
Correct Answer : 78
Model Answer   : 78
Correct?       : True
Time           : 1.553 seconds

Processing 4/50
Correct Answer : 8
Model Answer   : 8
Correct?       : True
Time           : 2.712 seconds

Processing 5/50
Correct Answer : 16
Model Answer   : 16
Correct?       : True
Time           : 3.87 seconds

Processing 6/50
Correct Answer : 183
Model Answer   : 183
Correct?       : True
Time           : 2.278 seconds

Processing 7/50
Correct Answer : 30
Model Answer   : 30
Correct?       : True
Time           : 2.088 seconds

Processing 8/50
Correct Answer : 12
Model Answer   : 12
Correct?       : True
Time           : 3.501 seconds

Processing 9/50
Correct Answer : 32
Model Answer   : 32
Correct?       : True
Time           : 1.68 seconds

Processing 

,Question_No,Question,Correct_Answer,Model,Model_Answer,Is_Correct,Response_Time_Seconds,Full_Response
0,1,Sarah has 24 apples. She gives 7 to her brothe...,32,o3-mini,32,True,8.032,"Starting with 24 apples, Sarah gives 7 away, l..."
1,2,A farmer has 35 chickens. He buys 18 more and ...,41,o3-mini,41,True,1.464,The farmer starts with 35 chickens. After buyi...
2,3,Tom reads 12 pages every day for 5 days. He th...,78,o3-mini,78,True,1.553,"Tom reads 12 pages per day for 5 days, which t..."
3,4,A box contains 48 pencils. The pencils are div...,8,o3-mini,8,True,2.712,We need to divide 48 pencils equally among 6 s...
4,5,Maria buys 4 notebooks at $3 each and 2 pens a...,16,o3-mini,16,True,3.870,"Maria buys 4 notebooks at $3 each, which costs..."
5,6,A school has 8 classrooms. Each classroom has ...,183,o3-mini,183,True,2.278,"There are 8 classrooms with 25 students each, ..."
6,7,John has $60. He spends $18 on a shirt and $12...,30,o3-mini,30,True,2.088,John starts with $60. He spends $18 on a shirt...
7,8,A baker makes 120 cookies. He packs them equal...,12,o3-mini,12,True,3.501,The baker packs the cookies evenly into 10 box...
8,9,Lisa walks 3 kilometers in the morning and 5 k...,32,o3-mini,32,True,1.680,Lisa walks 3 km in the morning and 5 km in the...
9,10,A store has 75 red balloons and 45 blue balloo...,82,o3-mini,82,True,1.995,"To solve, first add the red and blue balloons:..."


In [11]:
def ask_gpt_o4_mini(question):

    prompt = f"""
Solve the following GSM8K math problem.

Show your reasoning briefly and then provide the final answer.

At the very end, write the answer in this exact format:

FINAL ANSWER: <number>

Question:
{question}
"""

    start_time = time.perf_counter()

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )

    end_time = time.perf_counter()

    response_time = end_time - start_time

    full_response = response.output_text

    model_answer = extract_number(full_response)

    return full_response, model_answer, response_time

##

results = []

for i, row in df.iterrows():

    question_no = i + 1
    question = str(row["question"])
    correct_answer = str(row["answer"]).strip()

    print(f"\nProcessing {question_no}/{len(df)}")

    try:

        full_response, model_answer, response_time = ask_gpt_o4_mini(question)

        # Normalize answers before comparison
        correct_normalized = correct_answer.replace(",", "").strip()
        model_normalized = str(model_answer).replace(",", "").strip()

        # Compare numerically where possible
        try:
            is_correct = float(model_normalized) == float(correct_normalized)
        except:
            is_correct = model_normalized == correct_normalized

        results.append({
            "Question_No": question_no,
            "Question": question,
            "Correct_Answer": correct_answer,
            "Model": "gpt-4o-mini",
            "Model_Answer": model_answer,
            "Is_Correct": is_correct,
            "Response_Time_Seconds": round(response_time, 3),
            "Full_Response": full_response
        })

        print("Correct Answer :", correct_answer)
        print("Model Answer   :", model_answer)
        print("Correct?       :", is_correct)
        print("Time           :", round(response_time, 3), "seconds")

    except Exception as e:

        results.append({
            "Question_No": question_no,
            "Question": question,
            "Correct_Answer": correct_answer,
            "Model": "gpt-4o-mini",
            "Model_Answer": None,
            "Is_Correct": False,
            "Response_Time_Seconds": None,
            "Full_Response": f"ERROR: {str(e)}"
        })

        print("ERROR:", e)


# ==

results_df = pd.DataFrame(results)


# ===

total_questions = len(results_df)

correct_count = results_df["Is_Correct"].sum()

incorrect_count = total_questions - correct_count

accuracy = (correct_count / total_questions) * 100

average_time = results_df["Response_Time_Seconds"].mean()


# ====

print("\n")
print("=" * 60)
print("             GSM8K MODEL SCORECARD")
print("=" * 60)

print(f"Model              : gpt-4o-mini")
print(f"Total Questions    : {total_questions}")
print(f"Correct Answers    : {correct_count}")
print(f"Incorrect Answers  : {incorrect_count}")
print(f"Accuracy           : {accuracy:.2f}%")
print(f"Average Response   : {average_time:.3f} seconds")

print("=" * 60)


# ====

output_file = "gsm8k_gpt_o4_mini_results.csv"

results_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


# ====

display(results_df)

files.download(output_file)



Processing 1/50
Correct Answer : 32
Model Answer   : 32
Correct?       : True
Time           : 2.206 seconds

Processing 2/50
Correct Answer : 41
Model Answer   : 41
Correct?       : True
Time           : 2.847 seconds

Processing 3/50
Correct Answer : 78
Model Answer   : 78
Correct?       : True
Time           : 3.131 seconds

Processing 4/50
Correct Answer : 8
Model Answer   : 8
Correct?       : True
Time           : 2.216 seconds

Processing 5/50
Correct Answer : 16
Model Answer   : 16
Correct?       : True
Time           : 6.2 seconds

Processing 6/50
Correct Answer : 183
Model Answer   : 183
Correct?       : True
Time           : 1.586 seconds

Processing 7/50
Correct Answer : 30
Model Answer   : 30
Correct?       : True
Time           : 3.236 seconds

Processing 8/50
Correct Answer : 12
Model Answer   : 12
Correct?       : True
Time           : 2.109 seconds

Processing 9/50
Correct Answer : 32
Model Answer   : 32
Correct?       : True
Time           : 2.297 seconds

Processing 

,Question_No,Question,Correct_Answer,Model,Model_Answer,Is_Correct,Response_Time_Seconds,Full_Response
0,1,Sarah has 24 apples. She gives 7 to her brothe...,32,gpt-4o-mini,32,True,2.206,"To find out how many apples Sarah has now, we ..."
1,2,A farmer has 35 chickens. He buys 18 more and ...,41,gpt-4o-mini,41,True,2.847,"To solve the problem, we start with the number..."
2,3,Tom reads 12 pages every day for 5 days. He th...,78,gpt-4o-mini,78,True,3.131,"To find the total number of pages Tom read, we..."
3,4,A box contains 48 pencils. The pencils are div...,8,gpt-4o-mini,8,True,2.216,To find out how many pencils each student rece...
4,5,Maria buys 4 notebooks at $3 each and 2 pens a...,16,gpt-4o-mini,16,True,6.200,"To find out how much Maria spends in total, we..."
5,6,A school has 8 classrooms. Each classroom has ...,183,gpt-4o-mini,183,True,1.586,"To find the number of students present, we fir..."
6,7,John has $60. He spends $18 on a shirt and $12...,30,gpt-4o-mini,30,True,3.236,"To find out how much money John has left, we s..."
7,8,A baker makes 120 cookies. He packs them equal...,12,gpt-4o-mini,12,True,2.109,"To find out how many cookies are in each box, ..."
8,9,Lisa walks 3 kilometers in the morning and 5 k...,32,gpt-4o-mini,32,True,2.297,To find out how many kilometers Lisa walks in ...
9,10,A store has 75 red balloons and 45 blue balloo...,82,gpt-4o-mini,82,True,2.104,To find the number of balloons that remain aft...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>